In [ ]:
# ─── 1. CÀI ĐẶT MÔI TRƯỜNG (BẢN FIX LỖI NUMPY VÀ DATASETS) ───────────────────
!pip uninstall -y peft accelerate transformers datasets numpy pyarrow -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.20.0 \
  safetensors==0.4.2 \
  scikit-learn \
  peft==0.11.1 \
  Pillow \
  "numpy<2.0.0"

In [ ]:
# ─── 2. SETUP & IMPORTS ───────────────────────────────────────────────────────
import os, time, functools
import torch
import numpy as np
from PIL import Image
from torch import nn
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import (CLIPProcessor, CLIPModel, TrainingArguments,
                          Trainer, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset, Image as HFImage
from google.colab import drive

# Tối ưu VRAM
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Kết nối Drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/CLIP_ViT_B32_LoRA_IFND_Final"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save to:", SAVE_DIR)

# ─── 3. DATASET & PREPROCESS ──────────────────────────────────────────────────
print("⏳ Loading dataset IFND-multimodal...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")

# Ép kiểu dữ liệu ảnh để tránh lỗi ảnh Grayscale
hf_dataset = hf_dataset.cast_column("image", HFImage(decode=True))

model_name = "openai/clip-vit-base-patch32"
processor  = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    try:
        image = example['image'].convert("RGB")
    except:
        image = Image.new("RGB", (224, 224), (128, 128, 128))
    inputs = processor(
        text=str(example['text']),
        images=image,
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values":   inputs["pixel_values"][0],
        "labels":         int(example["label"])
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

train_dataset = encoded_dataset["train"]
val_dataset   = encoded_dataset["validation"]
test_dataset  = encoded_dataset["test"]
print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 4. MODEL ARCHITECTURE (LoRA + GATED FUSION + ALIGNMENT) ──────────────────
class CLIPForMFND(nn.Module):
    def __init__(self, model_name, num_labels=2, lambda_weight=0.1, delta=0.5):
        super().__init__()
        base_model = CLIPModel.from_pretrained(model_name)
        embed_dim = base_model.config.projection_dim # 512

        # 1. Cấu hình và tiêm LoRA
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=0.1,
            bias="none"
        )
        self.clip = get_peft_model(base_model, lora_config)

        # 2. Gated Fusion & Alignment Head
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=8, batch_first=True)
        self.W_g = nn.Linear(embed_dim * 2, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

        # 🛑 QUAN TRỌNG: Mở khóa Gradient cho Fusion Head (get_peft_model sẽ tự động khóa các layer không phải LoRA)
        for param in self.cross_attention.parameters(): param.requires_grad = True
        for param in self.W_g.parameters(): param.requires_grad = True
        for param in self.classifier.parameters(): param.requires_grad = True

        self.lambda_weight = lambda_weight
        self.delta = delta
        self.align_loss_fn = nn.CosineEmbeddingLoss(margin=delta)

    def forward(self, input_ids=None, attention_mask=None, pixel_values=None, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True
        )
        h_T, h_I = outputs.text_embeds, outputs.image_embeds

        # --- GATED MODALITY FUSION ---
        attn_out, _ = self.cross_attention(h_T.unsqueeze(1), h_I.unsqueeze(1), h_I.unsqueeze(1))
        z = attn_out.squeeze(1)

        gate = torch.sigmoid(self.W_g(torch.cat([h_T, h_I], dim=1)))
        h_f = gate * z

        # --- CLASSIFICATION & LOSS ---
        logits = self.classifier(h_f)
        loss = None
        if labels is not None:
            loss_cls = nn.CrossEntropyLoss()(logits, labels)
            loss_align = self.align_loss_fn(h_T, h_I, labels.float() * 2 - 1)
            loss = loss_cls + (self.lambda_weight * loss_align)

        return SequenceClassifierOutput(loss=loss, logits=logits)

import gc; gc.collect(); torch.cuda.empty_cache()

model = CLIPForMFND(model_name).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Total params    : {total_params:,} (~{total_params/1e6:.2f}M)")
print(f"📊 Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.4f}%)")

# ─── 5. METRICS & TRAINING ────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p,
        "recall":    r,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

torch.load = functools.partial(torch.load, weights_only=False)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=1e-4, # Tăng LR lên 1e-4 (LoRA hội tụ tốt hơn ở LR cao so với mức 2e-5 của FFT)
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🚀 Bắt đầu huấn luyện LoRA...")
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
start_train = time.time()

# Bỏ qua logic resume cũ, train đè luôn thư mục để tránh lỗi kiến trúc
trainer.train()

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
train_time_min = (time.time() - start_train) / 60
print(f"✅ Đã lưu mô hình tại: {SAVE_DIR} | Thời gian train: {train_time_min:.2f} phút")

# ─── 6. LATENCY + VRAM MEASUREMENT ────────────────────────────────────────────
model.eval()
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inputs = processor(
    text=dummy_text, images=dummy_image,
    truncation=True, padding="max_length",
    max_length=77, return_tensors="pt"
)
input_ids      = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values   = inputs["pixel_values"].to(device)

Using device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save to: /content/drive/MyDrive/CLIP_ViT_B32_LoRA_IFND_Final
⏳ Loading dataset IFND-multimodal...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


⏳ Preprocessing dataset...


Preprocessing:   0%|          | 0/1052 [00:00<?, ? examples/s]

✅ Train: 8416 | Val: 1052 | Test: 1053

📊 Total params    : 153,345,283 (~153.35M)
📊 Trainable params: 2,067,970 (1.3486%)

🚀 Bắt đầu huấn luyện LoRA...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.482700,0.237357,0.829848,0.818274,1.000000,0.900056,0.996929
2,0.071200,0.046018,0.992395,0.992593,0.997519,0.995050,0.997231
3,0.021700,0.032860,0.994297,0.993827,0.998759,0.996287,0.997193
4,0.015400,0.038912,0.994297,0.993827,0.998759,0.996287,0.997892
5,0.015100,0.038473,0.993346,0.993820,0.997519,0.995666,0.997703


✅ Đã lưu mô hình tại: /content/drive/MyDrive/CLIP_ViT_B32_LoRA_IFND_Final | Thời gian train: 15.23 phút


In [ ]:
# --- (Tiếp tục Phần 6) Đo lường Inference chuẩn (Batch Size = 1) ---
if device == "cuda": torch.cuda.synchronize()

# 1. Warmup (Cho GPU nóng máy)
with torch.no_grad():
    for _ in range(50):
        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

# 2. Đo đạc Latency
latencies = []
with torch.no_grad():
    for _ in range(200):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()

        model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**3 if device == "cuda" else 0

print(f"\n{'='*50}")
print(f"⚡ INFERENCE EFFICIENCY STATS (Batch Size = 1)")
print(f"Latency P50 (Median) : {np.median(latencies):.2f} ms")
print(f"Latency P90          : {np.percentile(latencies, 90):.2f} ms")
print(f"Latency Average      : {np.mean(latencies):.2f} ms")
print(f"VRAM Peak            : {vram:.2f} GB")
print(f"{'='*50}")

# ─── 7. FINAL EVALUATION & SAVE RESULTS TO JSON ───────────────────────────────
import json

print("\n📊 Evaluating on IFND TEST split...")
test_results = trainer.predict(test_dataset)
logits = test_results.predictions
labels = test_results.label_ids
probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds  = np.argmax(probs, axis=1)

p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
acc = accuracy_score(labels, preds)
auc = roc_auc_score(labels, probs[:, 1])

# Đóng gói kết quả
final_results = {
    "Model": "CLIP ViT-B/32",
    "Method": "LoRA (r=8)",
    "Dataset": "IFND-multimodal",
    "Hardware_Stats": {
        "Total_Params_M": round(total_params / 1e6, 2),
        "Trainable_Params_M": round(trainable_params / 1e6, 2),
        "Training_Time_Min": round(train_time_min, 2),
        "Peak_VRAM_GB": round(vram, 2),
        "Latency_P50_ms": round(np.median(latencies), 2),
        "Latency_P90_ms": round(np.percentile(latencies, 90), 2)
    },
    "Test_Metrics": {
        "Accuracy": round(acc * 100, 2),
        "Precision": round(p * 100, 2),
        "Recall": round(r * 100, 2),
        "F1_Score": round(f1 * 100, 2),
        "AUC": round(auc, 4)
    }
}

# Lưu ra file JSON
json_path = os.path.join(SAVE_DIR, "results_LoRA_IFND.json")
with open(json_path, "w") as f:
    json.dump(final_results, f, indent=4)

print(f"\n[TEST RESULT]")
print(f" Accuracy : {acc*100:.2f}%")
print(f" Precision: {p*100:.2f}%")
print(f" Recall   : {r*100:.2f}%")
print(f" F1 Score : {f1*100:.2f}% | AUC: {auc:.4f}")
print(f"\n💾 Đã lưu toàn bộ kết quả vào file: {json_path}")
print(f"{'='*50}")

# ─── 8. AUTO-DISCONNECT GPU ───────────────────────────────────────────────────
print("⏳ Đang ngắt kết nối GPU...")
from google.colab import runtime
time.sleep(5)
runtime.unassign()


⚡ INFERENCE EFFICIENCY STATS (Batch Size = 1)
Latency P50 (Median) : 75.22 ms
Latency P90          : 122.32 ms
Latency Average      : 87.62 ms
VRAM Peak            : 2.04 GB

📊 Evaluating on IFND TEST split...



[TEST RESULT]
 Accuracy : 99.53%
 Precision: 99.38%
 Recall   : 100.00%
 F1 Score : 99.69% | AUC: 0.9992

💾 Đã lưu toàn bộ kết quả vào file: /content/drive/MyDrive/CLIP_ViT_B32_LoRA_IFND_Final/results_LoRA_IFND.json
⏳ Đang ngắt kết nối GPU...
